In [0]:
%sql
SELECT 
    u.user_id,
    u.total_views,
    u.total_adds_to_cart,
    COUNT(f.sk_transaction) AS total_compras_realizadas,
    SUM(f.vlr_venda) AS total_gasto_pelo_cliente
FROM ecommerce_intelligence.gold.dim_users_activity u
LEFT JOIN ecommerce_intelligence.gold.fct_sales f 
    ON u.user_id = f.user_id
GROUP BY u.user_id, u.total_views, u.total_adds_to_cart
ORDER BY total_gasto_pelo_cliente DESC;

In [0]:
%sql

-- Taxa de abandono no carrinho
WITH metricas_base AS (
    SELECT 
        u.user_id,
        u.total_adds_to_cart,
        CASE WHEN COUNT(f.sk_transaction) > 0 THEN 1 ELSE 0 END AS realizou_compra    
    FROM ecommerce_intelligence.gold.dim_users_activity u
    LEFT JOIN ecommerce_intelligence.gold.fct_sales f 
        ON u.user_id = f.user_id
    GROUP BY u.user_id, u.total_adds_to_cart
)
SELECT
    COUNT(user_id) AS total_usuarios_com_carrinho,
    SUM(realizou_compra) AS total_usuarios_que_compraram,
    COUNT(user_id) - SUM(realizou_compra) AS total_usuarios_que_abandonaram,
    -- taxa de abandono
    round(
        (1 - (SUM(realizou_compra) / COUNT(user_id)))*100,
        2
    ) AS taxa_abandono
    FROM metricas_base;

In [0]:
%sql

-- Lista de usuários que abandonaram o carrinho
SELECT 
    u.user_id,
    u.total_views,       
    u.total_adds_to_cart,
    u.total_removes_from_cart,
    u.total_banner_clicks,
    u.gold_updated_timestamp as ultimo_acesso_ao_site    
FROM ecommerce_intelligence.gold.dim_users_activity u
LEFT JOIN ecommerce_intelligence.gold.fct_sales f 
    ON u.user_id = f.user_id
WHERE u.total_adds_to_cart > 0
  AND f.sk_transaction IS NULL
ORDER BY total_adds_to_cart;

In [0]:
%sql

-- ticket médio por pagamento
WITH faturamento_global AS (
    SELECT SUM(vlr_venda) AS faturamento_total_empresa FROM ecommerce_intelligence.gold.fct_sales
)
SELECT
  desc_metodo_pagamento,
  COUNT(sk_transaction) AS qtd_vendas,
  ROUND(SUM(vlr_venda), 2) AS faturamento_total,
  ROUND(SUM(vlr_venda) / COUNT(sk_transaction), 2) AS ticket_medio,
  ROUND((SUM(vlr_venda) / (SELECT faturamento_total_empresa FROM faturamento_global))*100, 2) AS pct_faturamento
FROM ecommerce_intelligence.gold.fct_sales
GROUP BY desc_metodo_pagamento
ORDER BY faturamento_total DESC;


In [0]:
%sql

--Análise de sazonalidade
SELECT
  CASE dayofweek(dt_transacao)
    WHEN 1 THEN 'Domingo'
    WHEN 2 THEN 'Segunda'
    WHEN 3 THEN 'Terça'
    WHEN 4 THEN 'Quarta'
    WHEN 5 THEN 'Quinta'
    WHEN 6 THEN 'Sexta'
    WHEN 7 THEN 'Sábado'
  END AS dia_semana,
  HOUR(dt_transacao) AS hora_dia,
  COUNT(sk_transaction) AS qtd_vendas,  
  ROUND(SUM(vlr_venda), 2) AS faturamento
FROM ecommerce_intelligence.gold.fct_sales
GROUP BY dayofweek(dt_transacao), hora_dia
ORDER BY dayofweek(dt_transacao), hora_dia; --faturamento_total DESC;


Databricks visualization. Run in Databricks to view.